In [ ]:
"""
FULL COLAB SCRIPT — V2 ASYMMETRIC, ALL CROSS-MODAL TASKS
(same as before — only CELL 8 plot_samples changed to overlay 300 samples)
"""

# ============================================================================
# CELL 1: SETUP
# ============================================================================

from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from tqdm import tqdm
from itertools import combinations

from scipy import stats
from sklearn.metrics import mean_squared_error, mean_absolute_error

print("✓ Setup complete!")
print(f"PyTorch: {torch.__version__}  |  CUDA: {torch.cuda.is_available()}")

# ============================================================================
# CELL 2: CONFIG
# ============================================================================

DATA_FILES = [
    "/content/drive/MyDrive/Multi_Modal_Rehab/synced/group_2_rotate_processed.csv",
    "/content/drive/MyDrive/Multi_Modal_Rehab/synced/group_3_rotate_processed.csv",
    "/content/drive/MyDrive/Multi_Modal_Rehab/synced/group_4_height_processed.csv",
]

BASE_SAVE_DIR    = "/content/drive/MyDrive/Multi_Modal_Rehab/cross_modal_v2"
SEQ_LENGTH       = 30
STRIDE           = 2
BATCH_SIZE       = 32
NUM_EPOCHS       = 150
PATIENCE         = 15
LEARNING_RATE    = 1e-3

PLOT_NUM_SAMPLES = 10
PLOT_CHANNEL     = 0

# ============================================================================
# CELL 3: V2 SPEC + TASK DEFINITIONS
# ============================================================================

V2 = {
    'eit': dict(input_size=8,  hidden_size=128, num_layers=3),
    'emg': dict(input_size=4,  hidden_size=64,  num_layers=2),
    'cap': dict(input_size=1,  hidden_size=32,  num_layers=1),
}
SHARED_SIZE    = 256
DECODER_HIDDEN = 128
DECODER_LAYERS = 2
DROPOUT        = 0.3

ALL_MODS = ['eit', 'emg', 'cap']

TASKS = []
for target in ALL_MODS:
    inputs = [m for m in ALL_MODS if m != target]
    TASKS.append({"inputs": inputs, "target": target,
                  "label": f"{'+'.join(inputs).upper()}→{target.upper()}", "type": "2→1"})

for src in ALL_MODS:
    for tgt in ALL_MODS:
        if src != tgt:
            TASKS.append({"inputs": [src], "target": tgt,
                          "label": f"{src.upper()}→{tgt.upper()}", "type": "1→1"})

print(f"Total tasks: {len(TASKS)}")

# ============================================================================
# CELL 4: HELPERS
# ============================================================================

def set_seed(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def safe_pearsonr(a, b):
    a, b = np.asarray(a).reshape(-1), np.asarray(b).reshape(-1)
    if a.size < 5 or np.std(a) < 1e-12 or np.std(b) < 1e-12:
        return 0.0
    return float(stats.pearsonr(a, b)[0])

def load_and_merge_sensor_dfs(file_list):
    dfs = []
    for fp in file_list:
        if not os.path.exists(fp):
            raise FileNotFoundError(f"Not found: {fp}")
        df = pd.read_csv(fp)
        df["source_file"] = os.path.basename(fp)
        keep = [c for c in df.columns
                if c.startswith("eit_ch") or c.startswith("emg") or c == "cap0_ma"]
        for extra in ["segment_id", "time_sec", "timestamp"]:
            if extra in df.columns:
                keep.append(extra)
        keep.append("source_file")
        dfs.append(df[keep].copy())
    merged = pd.concat(dfs, ignore_index=True)
    print(f"Merged: {merged.shape}  files: {merged['source_file'].unique().tolist()}")
    return merged

# ============================================================================
# CELL 5: MODEL
# ============================================================================

class ModalityEncoder(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout=0.2):
        super().__init__()
        self.hidden_size = hidden_size
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True,
                            dropout=dropout if num_layers > 1 else 0.0)
    def forward(self, x):
        _, (h, _) = self.lstm(x)
        return h[-1]


class FusionLayer(nn.Module):
    def __init__(self, fusion_input_size, shared_size=256, dropout=0.3):
        super().__init__()
        self.fc      = nn.Linear(fusion_input_size, shared_size)
        self.bn      = nn.BatchNorm1d(shared_size)
        self.dropout = nn.Dropout(dropout)
    def forward(self, feats):
        x = torch.cat(feats, dim=-1) if len(feats) > 1 else feats[0]
        return self.dropout(torch.relu(self.bn(self.fc(x))))


class LSTMDecoder(nn.Module):
    def __init__(self, input_size, hidden_size, output_size,
                 seq_length, num_layers=2, dropout=0.2):
        super().__init__()
        self.seq_length = seq_length
        self.fc_in  = nn.Linear(input_size, hidden_size)
        self.lstm   = nn.LSTM(hidden_size, hidden_size, num_layers, batch_first=True,
                              dropout=dropout if num_layers > 1 else 0.0)
        self.fc_out = nn.Linear(hidden_size, output_size)
    def forward(self, x):
        x = torch.relu(self.fc_in(x))
        x = x.unsqueeze(1).repeat(1, self.seq_length, 1)
        out, _ = self.lstm(x)
        return self.fc_out(out)


class V2CrossModalModel(nn.Module):
    def __init__(self, input_mods, target_mod, seq_length,
                 shared_size=256, decoder_hidden=128, decoder_layers=2, dropout=0.3):
        super().__init__()
        self.input_mods = list(input_mods)
        self.target_mod = target_mod
        self.encoders = nn.ModuleDict({
            mod: ModalityEncoder(V2[mod]['input_size'], V2[mod]['hidden_size'],
                                 V2[mod]['num_layers'], dropout)
            for mod in self.input_mods
        })
        fusion_in    = sum(V2[m]['hidden_size'] for m in self.input_mods)
        self.fusion  = FusionLayer(fusion_in, shared_size, dropout)
        self.decoder = LSTMDecoder(shared_size, decoder_hidden,
                                   V2[target_mod]['input_size'],
                                   seq_length, decoder_layers, dropout)
    def forward(self, batch):
        feats  = [self.encoders[m](batch[m]) for m in self.input_mods]
        shared = self.fusion(feats)
        return self.decoder(shared)

# ============================================================================
# CELL 6: DATASET
# ============================================================================

class CrossModalDataset(Dataset):
    def __init__(self, df, segment_ids=None, seq_length=30, stride=2, fill_value=0.0):
        self.seq_length = int(seq_length)
        self.stride     = int(stride)
        df = df.copy()
        eit_cols = [f"eit_ch{i}" for i in range(1, 9)]
        emg_cols = [f"emg{i}" for i in range(4)]
        cap_cols = ["cap0_ma"]
        has_seg  = "segment_id" in df.columns
        if segment_ids is not None:
            df = df[df["segment_id"].isin(segment_ids)].copy()
        if "time_sec" in df.columns:
            keys = ["source_file"] + (["segment_id"] if has_seg else []) + ["time_sec"]
            df = df.sort_values(keys).reset_index(drop=True)
        for c in eit_cols + emg_cols + cap_cols:
            df[c] = pd.to_numeric(df[c], errors="coerce")
        eit = df[eit_cols].to_numpy(float)
        emg = df[emg_cols].to_numpy(float)
        cap = df[cap_cols].to_numpy(float)
        valid = (np.isfinite(eit).any(1) | np.isfinite(emg).any(1) | np.isfinite(cap).any(1))
        df    = df[valid].reset_index(drop=True)
        def clean(arr):
            arr[np.isinf(arr)] = np.nan
            return np.nan_to_num(arr, nan=fill_value, posinf=fill_value, neginf=fill_value).astype(np.float32)
        eit = clean(df[eit_cols].to_numpy(np.float32))
        emg = clean(df[emg_cols].to_numpy(np.float32))
        cap = clean(df[cap_cols].to_numpy(np.float32))
        self.sequences = []
        for fname in df["source_file"].unique():
            if has_seg:
                for seg in df.loc[df["source_file"] == fname, "segment_id"].unique():
                    idx = df.index[(df["source_file"] == fname) & (df["segment_id"] == seg)].to_numpy()
                    self._window(eit[idx], emg[idx], cap[idx])
            else:
                idx = df.index[df["source_file"] == fname].to_numpy()
                self._window(eit[idx], emg[idx], cap[idx])
    def _window(self, eit, emg, cap):
        n = len(eit)
        if n < self.seq_length:
            return
        for i in range((n - self.seq_length) // self.stride + 1):
            s, e = i * self.stride, i * self.stride + self.seq_length
            self.sequences.append({"eit": eit[s:e], "emg": emg[s:e], "cap": cap[s:e]})
    def __len__(self):
        return len(self.sequences)
    def __getitem__(self, idx):
        s = self.sequences[idx]
        return {k: torch.tensor(v, dtype=torch.float32) for k, v in s.items()}

# ============================================================================
# CELL 7: TRAIN / VAL / EVAL
# ============================================================================

def train_epoch(model, loader, optimizer, device, target_key):
    model.train()
    criterion, losses = nn.MSELoss(), []
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        pred  = model(batch)
        loss  = criterion(pred, batch[target_key])
        if not torch.isfinite(loss):
            continue
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        losses.append(loss.item())
    return float(np.mean(losses)) if losses else 0.0


@torch.no_grad()
def validate_epoch(model, loader, device, target_key):
    model.eval()
    criterion, losses = nn.MSELoss(), []
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        loss  = criterion(model(batch), batch[target_key])
        if torch.isfinite(loss):
            losses.append(loss.item())
    return float(np.mean(losses)) if losses else 0.0


@torch.no_grad()
def evaluate(model, loader, device, target_key):
    model.eval()
    preds, gts = [], []
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        preds.append(model(batch).cpu().numpy())
        gts.append(batch[target_key].cpu().numpy())
    preds = np.concatenate(preds)
    gts   = np.concatenate(gts)
    p, t  = preds.reshape(-1), gts.reshape(-1)
    mse   = mean_squared_error(t, p)
    rmse  = float(np.sqrt(mse))
    mae   = float(mean_absolute_error(t, p))
    corr  = safe_pearsonr(p, t)
    ss_res = np.sum((t - p) ** 2)
    ss_tot = np.sum((t - t.mean()) ** 2)
    r2 = float(1 - ss_res / ss_tot) if ss_tot > 1e-12 else 0.0
    return preds, gts, {"mse": float(mse), "rmse": rmse, "mae": mae, "corr": corr, "r2": r2}


# ============================================================================
# CELL 8: PLOT SAMPLES  ← updated: overlay N samples in one figure per channel
#
#  Layout:
#    - One figure per output channel (C figures total)
#    - Each figure: True (top panel) vs Pred (bottom panel)
#      • N individual windows as thin semi-transparent lines
#      • Bold line = mean across all N windows
#      • Shaded band = ±1 std
#    - Saves one PNG per channel, shows inline in Colab
# ============================================================================

def plot_samples(preds, gts, label, save_dir, n_samples=200, seed=42):
    """
    preds, gts : [N_total, T, C]

    Concatenates n_samples windows end-to-end into one long signal,
    then plots True vs Predicted on the same axes per channel.
    Each window boundary is marked with a light dashed vertical line.

    One figure per channel, saved to save_dir.
    """
    os.makedirs(save_dir, exist_ok=True)

    N_total = preds.shape[0]
    T       = preds.shape[1]
    C       = preds.shape[2]
    n       = min(n_samples, N_total)

    # take n consecutive windows starting from a random offset
    # (consecutive = more natural long signal, avoids jumpy discontinuities)
    rng    = np.random.default_rng(seed)
    start  = rng.integers(0, max(1, N_total - n))
    idxs   = np.arange(start, start + n)

    # concatenate: [n*T] for each channel
    p_long = preds[idxs].reshape(-1, C)   # [n*T, C]
    g_long = gts[idxs].reshape(-1, C)     # [n*T, C]
    t_axis = np.arange(len(p_long))        # [n*T]

    for ch in range(C):
        fig, ax = plt.subplots(figsize=(12, 4))

        # window boundary lines
        for w in range(1, n):
            ax.axvline(x=w * T, color="#CCCCCC", linewidth=0.4, zorder=1)

        # true and predicted
        ax.plot(t_axis, g_long[:, ch],
                color="#185FA5", linewidth=0.8, alpha=0.85,
                label="True", zorder=3)
        ax.plot(t_axis, p_long[:, ch],
                color="#E24B4A", linewidth=0.8, alpha=0.85,
                linestyle="--", label="Predicted", zorder=4)

        ax.set_title(f"{label}  |  channel {ch}  |  {n} windows concatenated",
                     fontsize=12)
        ax.set_xlabel("Time step (concatenated windows)")
        ax.set_ylabel("Value")
        ax.legend(loc="upper right", fontsize=10)
        ax.grid(True, alpha=0.15, zorder=0)
        plt.tight_layout()

        fname    = label.replace('+', '_').replace('→', '_to_')
        out_path = os.path.join(save_dir, f"signal_{fname}_ch{ch}.png")
        plt.savefig(out_path, dpi=150, bbox_inches="tight")
        plt.show()
        plt.close()
        print(f"  Saved: {out_path}")


# ============================================================================
# CELL 9: MAIN — RUN ALL TASKS
# ============================================================================

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nDevice: {device}")

df_merged = load_and_merge_sensor_dfs(DATA_FILES)

full_ds = CrossModalDataset(df_merged, seq_length=SEQ_LENGTH, stride=STRIDE)
n       = len(full_ds)
n_train = int(0.70 * n)
n_val   = int(0.15 * n)
n_test  = n - n_train - n_val

train_ds, val_ds, test_ds = random_split(
    full_ds, [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(42)
)
train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True,  drop_last=True)
val_loader   = DataLoader(val_ds,   BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds,  BATCH_SIZE, shuffle=False)

print(f"Windows → train:{len(train_ds)}  val:{len(val_ds)}  test:{len(test_ds)}\n")

all_results = []

for task in TASKS:
    label      = task["label"]
    input_mods = task["inputs"]
    target_mod = task["target"]
    task_type  = task["type"]
    fusion_in  = sum(V2[m]['hidden_size'] for m in input_mods)

    print(f"\n{'='*60}")
    print(f"TASK: {label}  ({task_type})  fusion_in={fusion_in}")
    print(f"{'='*60}")

    save_dir  = os.path.join(BASE_SAVE_DIR, label.replace('+', '_').replace('→', '_to_'))
    os.makedirs(save_dir, exist_ok=True)

    model = V2CrossModalModel(
        input_mods=input_mods, target_mod=target_mod, seq_length=SEQ_LENGTH,
        shared_size=SHARED_SIZE, decoder_hidden=DECODER_HIDDEN,
        decoder_layers=DECODER_LAYERS, dropout=DROPOUT,
    ).to(device)

    n_params   = sum(p.numel() for p in model.parameters())
    optimizer  = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    scheduler  = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=10)
    best_val   = float("inf")
    no_improve = 0
    ckpt_path  = os.path.join(save_dir, "best_model.pth")

    for epoch in range(NUM_EPOCHS):
        tr = train_epoch(model, train_loader, optimizer, device, target_mod)
        va = validate_epoch(model, val_loader, device, target_mod)
        scheduler.step(va)
        if va < best_val:
            best_val, no_improve = va, 0
            torch.save({"epoch": epoch, "model": model.state_dict()}, ckpt_path)
        else:
            no_improve += 1
        if (epoch + 1) % 20 == 0 or no_improve == PATIENCE:
            print(f"  Ep {epoch+1:>3}  train={tr:.5f}  val={va:.5f}"
                  + (" ← best" if no_improve == 0 else f"  (no improve {no_improve}/{PATIENCE})"))
        if no_improve >= PATIENCE:
            print(f"  Early stop at epoch {epoch+1}"); break

    print(f"  Best val: {best_val:.6f}")
    model.load_state_dict(torch.load(ckpt_path, map_location=device)["model"])
    preds, gts, metrics = evaluate(model, test_loader, device, target_mod)
    print(f"  Test → MAE={metrics['mae']:.4f}  RMSE={metrics['rmse']:.4f}  "
          f"R2={metrics['r2']:.4f}  Corr={metrics['corr']:.4f}")

    # ── plot 300 overlaid samples ──
    plot_samples(preds, gts, label, save_dir, n_samples=PLOT_NUM_SAMPLES)

    all_results.append({
        "label": label, "type": task_type, "inputs": '+'.join(input_mods),
        "target": target_mod, "fusion_in": fusion_in, "params": n_params, **metrics
    })

# ============================================================================
# CELL 10: FINAL SUMMARY TABLE
# ============================================================================

df_res = pd.DataFrame(all_results)

print("\n" + "=" * 85)
print("FINAL SUMMARY — V2 ASYMMETRIC  (sorted by MAE)")
print("=" * 85)

for task_type in ["2→1", "1→1"]:
    subset = df_res[df_res["type"] == task_type].sort_values("mae")
    print(f"\n  {task_type} tasks:")
    print(f"  {'Label':<22} {'Inputs':<12} {'Target':<8} "
          f"{'Params':>10}  {'MAE':>8}  {'RMSE':>8}  {'R2':>7}  {'Corr':>7}")
    print("  " + "-" * 83)
    for _, row in subset.iterrows():
        print(f"  {row['label']:<22} {row['inputs']:<12} {row['target']:<8} "
              f"{row['params']:>10,}  {row['mae']:>8.4f}  {row['rmse']:>8.4f}  "
              f"{row['r2']:>7.4f}  {row['corr']:>7.4f}")

best_2to1 = df_res[df_res["type"]=="2→1"].sort_values("mae").iloc[0]
best_1to1 = df_res[df_res["type"]=="1→1"].sort_values("mae").iloc[0]
print(f"\n  Best 2→1: {best_2to1['label']:<22} MAE={best_2to1['mae']:.4f}")
print(f"  Best 1→1: {best_1to1['label']:<22} MAE={best_1to1['mae']:.4f}")

csv_path = os.path.join(BASE_SAVE_DIR, "cross_modal_summary_v2.csv")
df_res.to_csv(csv_path, index=False)
print(f"\n✓ Summary saved to: {csv_path}")